# BraTS-GoAT 2026 — Task 3 — Team NeuroAI

Clean pipeline. Run cells **in order**. Sections A–D are setup and must be
re-run in every new Kaggle session (`/tmp` is wiped when a session ends).

**Before you start:** Accelerator must be set to **GPU T4 x2**.
P100 fails — the installed PyTorch has no `sm_60` kernels.

| Section | What it does | Time |
|---|---|---|
| A | Install + verify | 2 min |
| B | Stage data (`.niigz` -> `.nii.gz`) | 2 min |
| C | Convert + preprocess | ~30 min |
| D | batch_dice patch + stratified splits | ~8 min |
| E | Train one fold | ~24 h (2 sessions) |
| F | Predict on validation set | 1.5-3 h |
| G | Verify + package for Synapse | 5 min |


## A — Install and verify

The numpy pin is **not optional**. Installing `nnunetv2` alone pulls numpy 2.5.x,
which breaks its own C extensions (`cannot import name '_center'`). Pinning in the
same pip command lets the resolver pick a numpy that satisfies both.

In [1]:
import glob
B = "/kaggle/working/nnUNet_results/Dataset501_GoAT/nnUNetTrainer_250epochs__nnUNetResEncUNetMPlans__3d_fullres"
for f in [0, 4]:
    print(f"fold {f}:", len(glob.glob(f"{B}/fold_{f}/validation/*.nii.gz")), "preds")

fold 0: 271 preds
fold 4: 270 preds


In [2]:
!pip install -q nnunetv2 nibabel "numpy>=2.0,<2.1" 

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 62.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 92.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 7.0 MB/s eta 0:00:00


In [3]:
import numpy as np
print("numpy", np.__version__)          # want 2.0.x
import nnunetv2
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
print("ok")                              # must print, else stop

numpy 2.0.2
ok


In [4]:
import os
os.environ["nnUNet_raw"]          = "/tmp/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/tmp/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/kaggle/working/nnUNet_results"
os.environ["nnUNet_n_proc_DA"]    = "4"      # 31 GB RAM; higher can OOM the host
print({k:v for k,v in os.environ.items() if k.startswith("nnUNet")})

{'nnUNet_raw': '/tmp/nnUNet_raw', 'nnUNet_preprocessed': '/tmp/nnUNet_preprocessed', 'nnUNet_results': '/kaggle/working/nnUNet_results', 'nnUNet_n_proc_DA': '4'}


In [5]:
#Training configuration table
import re, glob, json

log = sorted(glob.glob("/kaggle/working/nnUNet_results/**/fold_0/training_log_*.txt", recursive=True))[-1]
txt = open(log).read()

print("=== from training log ===")
for pat in [r"Using torch\.compile.*", r"do_dummy_2d_data_aug: \w+",
            r"This split has \d+ training and \d+ validation cases"]:
    m = re.search(pat, txt)
    if m: print(" ", m.group(0))

lrs = re.findall(r"Current learning rate: ([\d.e-]+)", txt)
eps = re.findall(r"Epoch (\d+)", txt)
ets = [float(x) for x in re.findall(r"Epoch time: ([\d.]+) s", txt)]
print(f"  initial LR: {lrs[0] if lrs else '?'}   final LR: {lrs[-1] if lrs else '?'}")
print(f"  epochs logged: {len(eps)}  last: {eps[-1] if eps else '?'}")
print(f"  epoch time: mean {sum(ets)/len(ets):.0f}s  total {sum(ets)/3600:.1f}h")

pl = json.load(open("/kaggle/input/datasets/aagnikraj/brats-t3/plans.json"))
c = pl["configurations"]["3d_fullres"]
print("\n=== from plans ===")
for k in ["patch_size","batch_size","batch_dice","normalization_schemes","spacing"]:
    print(f"  {k}: {c.get(k)}")

import torch, nnunetv2, numpy, sys
print(f"\n=== versions ===\n  python {sys.version.split()[0]}  torch {torch.__version__}")
print(f"  nnunetv2 {nnunetv2.__version__ if hasattr(nnunetv2,'__version__') else '2.5.1'}  numpy {numpy.__version__}")

=== from training log ===
  Using torch.compile... 
  do_dummy_2d_data_aug: False
  This split has 1080 training and 271 validation cases
  initial LR: 0.00235   final LR: 7e-05
  epochs logged: 50  last: 249
  epoch time: mean 298s  total 4.1h

=== from plans ===
  patch_size: [128, 160, 112]
  batch_size: 2
  batch_dice: True
  normalization_schemes: ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization']
  spacing: [1.0, 1.0, 1.0]

=== versions ===
  python 3.12.13  torch 2.10.0+cpu
  nnunetv2 2.5.1  numpy 2.0.2


In [6]:
#Convergence evidence
import re, glob
import numpy as np

log = sorted(glob.glob("/kaggle/working/nnUNet_results/**/fold_0/training_log_*.txt", recursive=True))[-1]
txt = open(log).read()
ema = [float(x) for x in re.findall(r"best EMA pseudo Dice: ([\d.]+)", txt)]
val = [float(x) for x in re.findall(r"val_loss ([-\d.]+)", txt)]

print(f"EMA Dice: first {ema[0]:.4f}  last {ema[-1]:.4f}")
print(f"  gain over final 50 improvements: {ema[-1]-ema[-min(50,len(ema))]:.4f}")
print(f"val_loss: last 25 mean {np.mean(val[-25:]):.4f}  prior 25 mean {np.mean(val[-50:-25]):.4f}")
print(f"  change: {np.mean(val[-25:])-np.mean(val[-50:-25]):+.4f}  (near 0 = plateaued)")

EMA Dice: first 0.9330  last 0.9355
  gain over final 50 improvements: 0.0025
val_loss: last 25 mean -0.9067  prior 25 mean -0.9017
  change: -0.0050  (near 0 = plateaued)


In [ ]:
#compared against conventional random cross-validation
#!/usr/bin/env python3
"""
Reviewer 6Y5g and B6Ya question: does the stratified split actually help,
vs. plain random 5-fold? Only needs cluster labels from goat_splits.py --
no GPU, no predictions.

Method: simulate 2000 random (unstratified) 5-fold splits, log the min
per-fold count for each minority cluster per trial, compare against the
guaranteed per-fold counts from the real stratified split.

Usage
    python stratified_vs_random.py
    (or paste into a Kaggle cell -- needs goat_cluster_labels.npy on disk,
     or hardcode the counts below from what's already been reported)
"""
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold

SEED = 42
N_TRIALS = 2000
N_FOLDS = 5

CLUSTER_NAMES = {0: "glioma-A", 3: "glioma-B", 1: "no-NCR",
                 2: "no-ET", 4: "ET-only/no-ED"}

# ---- load real cluster labels if available, else fall back to the
#      reported counts (525, 701, 67, 33, 25) so this still runs standalone
try:
    labels = np.load("/tmp/nnUNet_preprocessed/Dataset501_GoAT/goat_cluster_labels.npy")
    print(f"loaded real cluster labels, n={len(labels)}")
except FileNotFoundError:
    counts = {0: 525, 3: 701, 1: 67, 2: 33, 4: 25}
    labels = np.concatenate([np.full(n, k) for k, n in counts.items()])
    print(f"goat_cluster_labels.npy not found -- reconstructed from reported "
         f"counts, n={len(labels)}")

n = len(labels)
idx = np.arange(n)

# ---- the actual stratified split you used -------------------------------
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
strat_min = {k: N_FOLDS * [0] for k in CLUSTER_NAMES}
for fi, (_, val_idx) in enumerate(skf.split(idx, labels)):
    for k in CLUSTER_NAMES:
        strat_min[k][fi] = int((labels[val_idx] == k).sum())

print("\nACTUAL STRATIFIED SPLIT -- validation-fold counts per cluster")
print(f"{'cluster':>16} " + " ".join(f"fold{i}" for i in range(N_FOLDS)) + "   min")
for k in CLUSTER_NAMES:
    row = strat_min[k]
    print(f"{CLUSTER_NAMES[k]:>16} " + " ".join(f"{v:>5}" for v in row) +
          f"   {min(row):>3}")

# ---- Monte Carlo over plain (unstratified) random splits ----------------
rng = np.random.RandomState(SEED)
worst_case = {k: [] for k in CLUSTER_NAMES}   # min fold-count per trial
zero_fold_count = {k: 0 for k in CLUSTER_NAMES}  # trials with an EMPTY fold

for t in range(N_TRIALS):
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=rng.randint(1_000_000))
    for k in CLUSTER_NAMES:
        counts_this_trial = []
        for _, val_idx in kf.split(idx):
            counts_this_trial.append(int((labels[val_idx] == k).sum()))
        worst_case[k].append(min(counts_this_trial))
        if min(counts_this_trial) == 0:
            zero_fold_count[k] += 1

print(f"\nMONTE CARLO -- {N_TRIALS} random (unstratified) 5-fold splits")
print(f"{'cluster':>16} {'n':>5} {'mean min/fold':>15} {'worst seen':>11} "
     f"{'P(a fold=0)':>12}")
for k in CLUSTER_NAMES:
    n_k = int((labels == k).sum())
    wc = np.array(worst_case[k])
    p_zero = zero_fold_count[k] / N_TRIALS
    print(f"{CLUSTER_NAMES[k]:>16} {n_k:>5} {wc.mean():>15.2f} {wc.min():>11} "
         f"{p_zero:>11.1%}")

print("\nSUMMARY")
print("Stratified: every fold gets >=1 case per cluster, by construction.")
print("Random: some fraction of trials leave a fold with ZERO cases of the")
print("smallest clusters -- undefined score for that cluster on that fold,")
print("so no per-fold breakdown is possible on those splits.")

## B — Stage the data

Files are stored as `.niigz` so Kaggle does not auto-gunzip them on upload
(that inflates 22 GB to ~160 GB). They are still gzip inside. `nibabel` and
`SimpleITK` decide whether to decompress **by file extension**, so we create
symlinks named `.nii.gz` pointing at the `.niigz` files. No copying, no extra disk.

In [7]:
from pathlib import Path

STAGE = Path("/tmp/staged"); STAGE.mkdir(parents=True, exist_ok=True)
n = 0
for root in Path("/kaggle/input").glob("*"):
    for f in root.rglob("*.niigz"):
        dst = STAGE / root.name / f.relative_to(root).with_suffix("").with_suffix(".nii.gz")
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.is_symlink() or dst.exists():
            dst.unlink()
        dst.symlink_to(f.resolve())
        n += 1
print(n, "files linked   (expect 8559)")

8559 files linked   (expect 8559)


Path discovery. Kaggle prefixes the mount with the dataset owner's slug, so the
paths differ per account. Run this and paste the two results into the next cell.

In [8]:
!find /tmp/staged -maxdepth 6 -type d -name "*With-GroundTruth" | tail -1
!find /tmp/staged -maxdepth 6 -type d -name "*ValidationData"    | tail -1

/tmp/staged/datasets/aagnikraj/brats-t3/MICCAI2024-BraTS-GoAT-TrainingData-With-GroundTruth/MICCAI2024-BraTS-GoAT-TrainingData-With-GroundTruth
/tmp/staged/datasets/aagnikraj/brats-t3/MICCAI2024-BraTS-GoAT-ValidationData/MICCAI2024-BraTS-GoAT-ValidationData


## C — Convert to nnU-Net format, then preprocess

`goat_to_nnunet.py` and `goat_splits.py` must already be in `/kaggle/working/`.
Get them from the `goat-handover` dataset or fork the notebook that has them.

`--verify 40` round-trips 40 ground-truth masks through the region encoding and
asserts they rebuild exactly. It catches a wrong `regions_class_order` before you
spend 24 h training against it. Cheap insurance — leave it on.

In [9]:
TRAIN_GT = "/tmp/staged/datasets/aagnikraj/brats-t3/MICCAI2024-BraTS-GoAT-TrainingData-With-GroundTruth/MICCAI2024-BraTS-GoAT-TrainingData-With-GroundTruth"
VAL_DIR  = "/tmp/staged/datasets/aagnikraj/brats-t3/MICCAI2024-BraTS-GoAT-ValidationData/MICCAI2024-BraTS-GoAT-ValidationData"

!python /kaggle/working/goat_to_nnunet.py \
  --train-gt "{TRAIN_GT}" \
  --val      "{VAL_DIR}" \
  --out /tmp/nnUNet_raw --verify 40

# expect: 1351 cases (expect 1351)  /  451 cases (expect 451)  /  round-trip OK

[1/3] linking training cases -> /tmp/nnUNet_raw/Dataset501_GoAT/imagesTr
      1351 cases (expect 1351)
[2/3] linking validation cases -> /tmp/nnUNet_raw/Dataset501_GoAT/imagesTs
      451 cases (expect 451)
[3/3] wrote /tmp/nnUNet_raw/Dataset501_GoAT/dataset.json
  [verify] round-trip OK on 40 cases
  [verify] absent: NCR=3/40, ED=2/40, ET=2/40

next:
  export nnUNet_raw=/tmp/nnUNet_raw


Preprocessing: crops to brain, resamples to 1 mm isotropic, z-scores each
modality, writes ~30-45 GB into `/tmp`. Takes ~30 min. It is CPU-bound, so no
GPU is consumed. **No live output** until it finishes — that is normal.

In [10]:
import subprocess, os
cmd = ["nnUNetv2_plan_and_preprocess", "-d", "501",
       "-pl", "nnUNetPlannerResEncM", "-c", "3d_fullres", "-np", "4"]
p = subprocess.Popen(cmd, env=os.environ, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)

Fingerprint extraction...
Dataset501_GoAT
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer

Extracting dataset fingerprint: 100%|██████████| 1351/1351 [10:07<00:00,  2.22it/s]
Experiment planning...
Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [140. 171. 136.], 3d_lowres: [140, 171, 136]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 107, 'patch_size': (np.int64(192), np.int64(160)), 'median_image_size_in_voxels': array([171., 136.]), 'spacing': array([1., 1.]), 'normalization_schemes': ['ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization', 'ZScoreNormalization'], 'use_mask_for_norm': [True, True, True, True], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': Non

## D — batch_dice patch + stratified splits

**Both are mandatory and both are destroyed by re-running section C**, because
`plan_and_preprocess` rewrites the plans file. Always re-apply after preprocessing.

**`batch_dice=True`** computes Dice over the pooled batch instead of per sample.
About 2.4% of cases have no enhancing tumour at all; per-sample Dice is degenerate
on those. It also matches the leaderboard, which pools voxels across the whole
cohort before computing one score.

**Stratified splits**: case IDs are opaque (`BraTS-GoAT-01234`), so a random 5-fold
can dump nearly all of a rare cohort into one fold. `goat_splits.py` fingerprints
each case (sub-region presence, volumes, brain bbox, intensity percentiles),
k-means into 5 appearance groups, then stratifies. Seed is fixed at 42 —
**everyone gets identical folds, which is what makes the ensemble valid.**

In [11]:
import json
P = "/tmp/nnUNet_preprocessed/Dataset501_GoAT/nnUNetResEncUNetMPlans.json"
pl = json.load(open(P))
pl["configurations"]["3d_fullres"]["batch_dice"] = True
json.dump(pl, open(P, "w"), indent=4)
c = pl["configurations"]["3d_fullres"]
print("batch_dice:", c["batch_dice"], "| patch:", c["patch_size"], "| batch:", c["batch_size"])
# expect: True | [128, 160, 112] | 2

batch_dice: True | patch: [128, 160, 112] | batch: 2


In [12]:
!python /kaggle/working/goat_splits.py \
  --raw /tmp/nnUNet_raw/Dataset501_GoAT \
  --preprocessed /tmp/nnUNet_preprocessed/Dataset501_GoAT \
  --k 5 --folds 5

[1/4] fingerprinting 1351 cases
      200/1351
      400/1351
      600/1351
      800/1351
      1000/1351
      1200/1351
[2/4] k-means, k=5

      cluster   n    NCR+    ED+    ET+   med_ET_frac
         0     525   1.00   1.00   1.00   0.220
         1      67   0.00   1.00   1.00   0.161
         2      33   0.82   1.00   0.00   0.000
         3     701   1.00   1.00   1.00   0.223
         4      25   0.08   0.00   1.00   1.000

[3/4] stratified 5-fold over clusters
      fold 0: 271 val, cluster mix [105, 14, 6, 141, 5]
      fold 1: 270 val, cluster mix [105, 13, 7, 140, 5]
      fold 2: 270 val, cluster mix [105, 13, 7, 140, 5]
      fold 3: 270 val, cluster mix [105, 13, 7, 140, 5]
      fold 4: 270 val, cluster mix [105, 14, 6, 140, 5]

[4/4] wrote /tmp/nnUNet_preprocessed/Dataset501_GoAT/splits_final.json


In [13]:
import json
s = json.load(open("/tmp/nnUNet_preprocessed/Dataset501_GoAT/splits_final.json"))
print(len(s[2]["train"]), len(s[2]["val"]))   # must be 1081 270

1081 270


### Gate — do not train until both are True

In [14]:
import json, os
P = "/tmp/nnUNet_preprocessed/Dataset501_GoAT/nnUNetResEncUNetMPlans.json"
print("batch_dice:", json.load(open(P))["configurations"]["3d_fullres"]["batch_dice"])
print("splits    :", os.path.exists("/tmp/nnUNet_preprocessed/Dataset501_GoAT/splits_final.json"))

batch_dice: True
splits    : True


## E — Train one fold

**Set `FOLD` to your assigned number.** Aagnik already did fold 0.

Never use `%%bash` for this — it buffers all output and shows nothing until the
process exits, which looks identical to a hang. `subprocess.Popen` streams live.

- ~290-350 s/epoch on one T4, 250 epochs -> **~24 h = 2 sessions**
- nnU-Net checkpoints every 50 epochs to `/kaggle/working`
- **Hit Save Version before the 12 h wall** or `/kaggle/working` is wiped
- Next session: re-run A-D (~40 min) then this cell — `--c` resumes automatically

In [ ]:
FOLD = "4"          # <<<<<< CHANGE THIS

import subprocess, os
cmd = ["nnUNetv2_train", "501", "3d_fullres", FOLD,
       "-p", "nnUNetResEncUNetMPlans",
       "-tr", "nnUNetTrainer_250epochs",
       "--npz",      # saves softmax; needed for ensembling and threshold tuning
       "--c"]        # resume if a checkpoint exists; harmless on a fresh start
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)

### Check what has been saved (run in a separate cell any time)

In [ ]:
import torch, os, glob
FOLD = "4"
B = f"/kaggle/working/nnUNet_results/Dataset501_GoAT/nnUNetTrainer_250epochs__nnUNetResEncUNetMPlans__3d_fullres/fold_{FOLD}"
for f in ["checkpoint_latest.pth", "checkpoint_best.pth", "checkpoint_final.pth"]:
    p = os.path.join(B, f)
    if os.path.exists(p):
        ck = torch.load(p, map_location="cpu", weights_only=False)
        print(f"{f:24s} epoch={ck.get('current_epoch','?')}")
    else:
        print(f"{f:24s} MISSING")
print("npz:", len(glob.glob(f"{B}/validation/*.npz")))

In [ ]:
import glob, os, shutil
SRC = ("/kaggle/working/nnUNet_results/Dataset501_GoAT/"
       "nnUNetTrainer_250epochs__nnUNetResEncUNetMPlans__3d_fullres/fold_0/validation")
DST = "/kaggle/working/goat_fold0_val_preds"
os.makedirs(DST, exist_ok=True)

preds = sorted(glob.glob(f"{SRC}/*.nii.gz"))
print("found:", len(preds), "(expect 271)")
for f in preds:
    shutil.copy(f, DST)
print("MB:", round(sum(os.path.getsize(f"{DST}/{f}") for f in os.listdir(DST))/1e6, 1))

## F — Predict on the 451 validation cases

Only needed by whoever assembles the submission.

Inference reads `plans.json` / `dataset.json` from `nnUNet_results`, **not** from
`/tmp` — so this does not require section C's preprocessing, only section B's
staging plus the `imagesTs` links built below.

### The settings that matter
- `-f 0 1 2 3 4` averages softmax across folds **before** argmax. This is the
  ensemble. With one fold, use `-f 0`.
- `-step_size 0.25` = 75% sliding-window overlap (default 0.5 = 50%). Denser
  windows mean smoother, better-localised boundaries. **NSD is 3 of the 6 ranking
  columns and the whole field under-optimises it.** Costs ~2-3x inference time,
  changes nothing about the model. Free rank.
- Mirroring TTA and Gaussian window weighting are **on by default**. Do not pass
  `--disable_tta`.
- `-chk checkpoint_final.pth` — the fully annealed epoch-250 weights, normally
  better for prediction than the EMA `checkpoint_best`.

In [ ]:
# build imagesTs from the validation cases (channel order is frozen: t1n,t1c,t2w,t2f)
import os
VAL_DIR  = "/tmp/staged/datasets/aagnikraj/brats-t3/MICCAI2024-BraTS-GoAT-ValidationData/MICCAI2024-BraTS-GoAT-ValidationData"
imagesTs = "/tmp/nnUNet_raw/Dataset501_GoAT/imagesTs"
os.makedirs(imagesTs, exist_ok=True)

CH = {0: "t1n", 1: "t1c", 2: "t2w", 3: "t2f"}
cnt = 0
for case in sorted(os.listdir(VAL_DIR)):
    if not case.startswith("BraTS-GoAT-"):
        continue
    for idx, suf in CH.items():
        src = f"{VAL_DIR}/{case}/{case}-{suf}.nii.gz"
        dst = f"{imagesTs}/{case}_{idx:04d}.nii.gz"
        if os.path.islink(dst) or os.path.exists(dst):
            os.remove(dst)
        os.symlink(os.path.realpath(src), dst)
        cnt += 1
print("linked", cnt, "(expect 1804)")

In [ ]:
import subprocess, os

FOLDS = ["0"]                 # ensemble: ["0","1","2","3","4"]
STEP  = "0.25"                # 0.5 = default/faster, 0.25 = denser, better NSD

cmd = ["nnUNetv2_predict",
       "-i", "/tmp/nnUNet_raw/Dataset501_GoAT/imagesTs",
       "-o", "/tmp/goat_val_pred",
       "-d", "501", "-c", "3d_fullres",
       "-f", *FOLDS,
       "-p", "nnUNetResEncUNetMPlans",
       "-tr", "nnUNetTrainer_250epochs",
       "-chk", "checkpoint_final.pth",
       "-step_size", STEP]
env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="", flush=True)

## G — Verify, then package

The rules say mismatched geometry causes **submission invalidation**, so check
before zipping, not after.

In [ ]:
import os, glob, re, nibabel as nib, numpy as np

pred_dir = "/tmp/goat_val_pred"
src_dir  = "/tmp/nnUNet_raw/Dataset501_GoAT/imagesTs"
preds = sorted(glob.glob(f"{pred_dir}/*.nii.gz"))
print("predictions:", len(preds), "(expect 451)")

bad = [os.path.basename(p) for p in preds
       if not re.fullmatch(r"BraTS-GoAT-\d{5}\.nii\.gz", os.path.basename(p))]
print("badly named:", bad if bad else "none")

ok = len(preds) == 451 and not bad
for i in [0, len(preds)//4, len(preds)//2, 3*len(preds)//4, len(preds)-1]:
    cid = os.path.basename(preds[i])[:-7]
    pi = nib.load(preds[i]); si = nib.load(f"{src_dir}/{cid}_0000.nii.gz")
    s_ok = pi.shape == si.shape
    a_ok = np.allclose(pi.affine, si.affine, atol=1e-4)
    labs = np.unique(np.asarray(pi.dataobj)).tolist()
    l_ok = set(labs).issubset({0, 1, 2, 3})
    print(f"{cid}: shape={'OK' if s_ok else 'BAD'} affine={'OK' if a_ok else 'BAD'} labels={labs}")
    ok &= s_ok and a_ok and l_ok

print("\nSAFE TO ZIP" if ok else "\nPROBLEM - do not submit")

### Packaging — this is where the first submission was rejected

`shutil.make_archive` sweeps the **whole folder**, and nnU-Net leaves
`dataset.json`, `plans.json` and `predict_from_raw_data_args.json` in the output
directory. The validator rejected it: *"Not all files in the archive are NIfTI
files."*

Zip the `.nii.gz` files explicitly, flat (`arcname=basename`, no sub-folders).

In [ ]:
import glob, os, zipfile

preds = sorted(glob.glob("/tmp/goat_val_pred/*.nii.gz"))
out = "/kaggle/working/goat_task3_submission.zip"
if os.path.exists(out):
    os.remove(out)

with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in preds:
        z.write(p, arcname=os.path.basename(p))   # flat, no directories

with zipfile.ZipFile(out) as z:
    names = z.namelist()
print("entries      :", len(names), "(expect 451)")
print("all .nii.gz  :", all(n.endswith(".nii.gz") for n in names))
print("non-nifti    :", [n for n in names if not n.endswith(".nii.gz")] or "none")
print("size MB      :", round(os.path.getsize(out)/1e6))

All four lines must be clean, then **Save Version**, download the zip from the
Output tab, and upload to Synapse -> Submit File -> Task 3 queue.

Filename rule for Task 3: `BraTS-GoAT-XXXXX.nii.gz`, case ID only, **no**
timepoint (that requirement is waived for GoAT). nnU-Net already names them
correctly — no renaming needed.

# More stuff


In [15]:
# ============================================================
# PER-FOLD x CLUSTER ANALYSIS  — 4 folds (0,1,2,4), fold 3 pending
# ============================================================
import json, glob, csv
import numpy as np, nibabel as nib
from pathlib import Path
from collections import defaultdict

PREP = Path("/tmp/nnUNet_preprocessed/Dataset501_GoAT")
GT   = Path("/tmp/nnUNet_raw/Dataset501_GoAT/labelsTr")
REG  = {"WT": (1,2,3), "TC": (1,3), "ET": (3,)}
NAMES = {0:"glioma-A", 3:"glioma-B", 1:"no-NCR", 2:"no-ET", 4:"ET-only/no-ED"}

FOLD_PATHS = {
    0: "/kaggle/working/nnUNet_results/Dataset501_GoAT/nnUNetTrainer_250epochs__nnUNetResEncUNetMPlans__3d_fullres/fold_0/validation",
    4: "/kaggle/working/nnUNet_results/Dataset501_GoAT/nnUNetTrainer_250epochs__nnUNetResEncUNetMPlans__3d_fullres/fold_4/validation",
    1: "/kaggle/input/datasets/aagnikraj/brats-t3/fold1_validation/validation",
    2: "/kaggle/input/datasets/aagnikraj/brats-t3/fold2_validation/validation",
    # 3: not yet available
}

def find_pred(pdir, cid):
    """Predictions may be .nii.gz or .nii depending on how the fold was zipped."""
    for ext in (".nii.gz", ".nii"):
        p = pdir / f"{cid}{ext}"
        if p.exists():
            return p
    return None

cl_arr = np.load(PREP/"goat_cluster_labels.npy")
cids   = sorted(p.name[:-7] for p in GT.glob("*.nii.gz"))
assert len(cl_arr) == len(cids), "cluster/labelsTr mismatch"
cluster = dict(zip(cids, cl_arr.tolist()))
splits  = json.load(open(PREP/"splits_final.json"))
load    = lambda p: np.asarray(nib.load(str(p)).dataobj).astype(np.uint8)

pooled  = defaultdict(lambda: [0,0,0])
percase = defaultdict(list)
rows, missing = [], 0

for f, path in FOLD_PATHS.items():
    if path is None:
        print(f"\n[skip] fold {f}: path not set yet")
        continue
    pdir = Path(path)
    if not pdir.exists():
        print(f"\n[skip] fold {f}: {pdir} does not exist")
        continue
    val = splits[f]["val"]
    print(f"\nfold {f}: scoring {len(val)} cases from {pdir} ...", flush=True)
    n_scored = 0
    for n, cid in enumerate(val):
        pf = find_pred(pdir, cid)
        if pf is None:
            missing += 1
            continue
        pr, gt = load(pf), load(GT/f"{cid}.nii.gz")
        k = cluster[cid]
        row = {"case": cid, "fold": f, "cluster": k}
        for rname, labs in REG.items():
            p, g = np.isin(pr, labs), np.isin(gt, labs)
            tp, fp, fn = int((p&g).sum()), int((p&~g).sum()), int((~p&g).sum())
            a = pooled[(f,k,rname)]; a[0]+=tp; a[1]+=fp; a[2]+=fn
            d = 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else np.nan
            if not np.isnan(d): percase[(k,rname)].append(d)
            row[f"dice_{rname}"] = round(d,4) if not np.isnan(d) else ""
        rows.append(row)
        n_scored += 1
        if (n+1) % 100 == 0: print(f"   {n+1}/{len(val)}", flush=True)
    print(f"   scored {n_scored}/{len(val)}")

if missing: print(f"\n[warn] {missing} predictions missing across all folds")

def dice(t): return 2*t[0]/(2*t[0]+t[1]+t[2]) if (2*t[0]+t[1]+t[2]) else np.nan
folds_present = sorted({r["fold"] for r in rows})
print(f"\nfolds actually scored: {folds_present}")

print("\n\nPOOLED DICE BY CLUSTER (folds scored so far combined)")
print(f"{'cluster':>16} {'n':>5} {'WT':>8} {'TC':>8} {'ET':>8}")
for k in [0,3,1,2,4]:
    n = sum(1 for r in rows if r["cluster"]==k)
    if not n: continue
    line = f"{NAMES[k]:>16} {n:>5}"
    for rname in ("WT","TC","ET"):
        agg = [sum(pooled[(f,k,rname)][i] for f in folds_present) for i in range(3)]
        line += f" {dice(agg):>8.4f}"
    print(line)

print("\nPER-CASE MEAN +/- SD BY CLUSTER")
for k in [0,3,1,2,4]:
    parts = []
    for rname in ("WT","TC","ET"):
        v = np.array(percase[(k,rname)])
        parts.append(f"{rname} {v.mean():.3f}+/-{v.std():.3f} (n={len(v)})" if len(v) else f"{rname} n/a")
    print(f"{NAMES[k]:>16}  " + " | ".join(parts))

print(f"\nPER-FOLD WT DICE BY CLUSTER  (stability check, folds {folds_present})")
print(f"{'cluster':>16} " + " ".join(f"{'fold'+str(f):>8}" for f in folds_present))
for k in [0,3,1,2,4]:
    print(f"{NAMES[k]:>16} " + " ".join(f"{dice(pooled[(f,k,'WT')]):>8.4f}" for f in folds_present))

with open("/kaggle/working/per_fold_cluster.csv","w",newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=["case","fold","cluster","dice_WT","dice_TC","dice_ET"])
    w.writeheader(); w.writerows(rows)
print(f"\nwrote per_fold_cluster.csv ({len(rows)} rows)")
print(f"\n{'='*50}\nMissing: fold 3. Re-run once available — only")
print(f"FOLD_PATHS needs updating, nothing else.\n{'='*50}")


fold 0: scoring 271 cases from /kaggle/working/nnUNet_results/Dataset501_GoAT/nnUNetTrainer_250epochs__nnUNetResEncUNetMPlans__3d_fullres/fold_0/validation ...
   100/271
   200/271
   scored 271/271

fold 4: scoring 270 cases from /kaggle/working/nnUNet_results/Dataset501_GoAT/nnUNetTrainer_250epochs__nnUNetResEncUNetMPlans__3d_fullres/fold_4/validation ...
   100/270
   200/270
   scored 270/270

fold 1: scoring 270 cases from /kaggle/input/datasets/aagnikraj/brats-t3/fold1_validation/validation ...
   100/270
   200/270
   scored 270/270

fold 2: scoring 270 cases from /kaggle/input/datasets/aagnikraj/brats-t3/fold2_validation/validation ...
   scored 13/270

[warn] 257 predictions missing across all folds

folds actually scored: [0, 1, 2, 4]


POOLED DICE BY CLUSTER (folds scored so far combined)
         cluster     n       WT       TC       ET
        glioma-A   317   0.9560   0.9537   0.9229
        glioma-B   428   0.9422   0.9309   0.9168
          no-NCR    43   0.9038   0.9